# Install GNINA

In [ ]:
#Import Cuda and Gnina
!nvidia-smi
!wget https://github.com/gnina/gnina/releases/download/v1.0.3/gnina
!chmod +x gnina

Thu Oct  2 11:48:33 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Make it executable
!chmod +x gnina



# How to Prepare Input
Here is an example of how to use the "AutoBox" feature of GNINA. To do this, the tool here in the vanilla version simply needs a PDB file with bound ligand (which is then to be replaced). In the example here I use the PDB structure 3MXF which uses a bromodomain from BRD4 with bound JQ1. The docking tool requires three files:

1. Receiver only
2. Ligand only (the one which should be replaced)
3. Ligand only (the one which should be docked into the pocket)
The first two were generated in the next cells based on your input PDB fell. The whole protein is always pulled out of the PDB file and pushed into the file "rec.pdb". To prevent the wrong ligand from being pulled out, the script needs three parameters:

(example)

1. Ligand ID (JQ1)
2. Ligand Chain (A)
3. Ligand Position (1)
The easiest way to get this info is to upload your PDB file to PLIP (https://plip-tool.biotec.tu-dresden.de/plip-web/plip/index) and look at the Binding side info on the left to which chain + position your ligand binds. But you can also look at the PDB file and find everything in the HEATM entries. This is automatically written to a file "orig.pdb".

The ligands you want to dock must be as a .sdf file. It either comes from a database (e.g. PubChem) or you create the file from a Smiles string with the helper script in the next cell.

#Optional: Helper to create a ligand structure file .sdf from smiles string

In [ ]:
#title Install RDKIT in colab
try:
  from rdkit import Chem
  from rdkit.Chem import AllChem
except:
  !pip install rdkit
  from rdkit import Chem
  from rdkit.Chem import AllChem


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.1/36.1 MB 65.5 MB/s eta 0:00:00


In [ ]:
#@title Define Ligand to dock + Output File path (Optional)
Ligand_Smiles = 'Cc1sc2c(c1C)C(=N[CH](CC(=O)OC(C)(C)C)c3[nH]nc(C)[n+]23)c4ccc(Cl)cc4' # @param {type:"string"}
Ligand_File = '/content/Ligand_to_dock.sdf' # @param {type:"string"}

In [ ]:
#@title Create the ligand file
def smiles_to_sdf(smiles: str, out_path: str, name: str = "Ligand"):
    """
    Convert a SMILES string to an SDF file with 3D coordinates.

    Args:
        smiles: SMILES string of the molecule
        out_path: output .sdf filename
        name: optional molecule name (stored in the SDF)
    """
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)  # add hydrogens
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())  # generate 3D coords
    AllChem.UFFOptimizeMolecule(mol)  # optional: energy minimize
    mol.SetProp("_Name", name)

    w = Chem.SDWriter(out_path)
    w.write(mol)
    w.close()
    print(f"Wrote {out_path}")

smiles_to_sdf(smiles=Ligand_Smiles, out_path=Ligand_File)

Wrote /content/Ligand_to_dock.sdf


In [ ]:
#@title Docking with autobox ligand (Not Optional any more)
Protein_File = '/content/3MXF.pdb' # @param {type:"string"}
Ligand_File = '/content/Ligand_to_dock.sdf' # @param {type:"string"}

# Ligand used to define binding box from input complex
orig_ligand = 'JQ1:A:1' # @param {type:"string"}

In [ ]:
#@title Prepare output folder
!mkdir -p Gnina_output
!mkdir -p Gnina_output_orig_ligand

# Prepare all required input files

lig_grep = ' *'.join(orig_ligand.split(':'))
print(lig_grep)

!grep ATOM $Protein_File > rec.pdb
!grep HETATM $Protein_File | grep "$lig_grep" > orig.pdb

JQ1 *A *1


In [ ]:
#@title Do the actual docking using the autobox flag without further constrains
!./gnina -r rec.pdb -l $Ligand_File --autobox_ligand orig.pdb   --out ./Gnina_output/docked_ligand.sdf --log ./Gnina_output/docking.log

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina  master:e9cb230+   Built Feb 11 2023.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: ./gnina -r rec.pdb -l /content/Ligand_to_dock.sdf --autobox_ligand orig.pdb --out ./Gnina_output/docked_ligand.sdf --log ./Gnina_output/docking.log
Using random seed: -235944064

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
Ligand | pose 0 | initial pose not within box

mode |  affinity  |    CNN     |   CNN
     | (kcal/mol) | pose score | affinity
-----+------------+------------+----------
    1       -8.24       0.8658      6.944
    2       -8.15       0.7559      6.972
    3       -6.54       0.4447      6.321
    

## Create complexes for the docking poses
This part is to generate full complexes out of the GNINA output. There is to
"good" way to do it but the script I wrote here does work most of the times :D.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from typing import Dict, List, Tuple, Iterable, Optional
from collections import defaultdict
from rdkit import Chem

# ---------------- PDB formatting helpers ----------------

_TWO_LETTER = {"BR","CL","FE","ZN","MG","NA","CA","MN","CO","NI","CU","AL","SI","KR","XE","SR","CD","HG","PB","SN"}

def _ensure_nl80(line: str) -> str:
    if not line.endswith("\n"):
        line += "\n"
    # PDB records are 80 columns; keep or pad to 80 + newline
    if len(line) < 81:
        line = line[:-1] + " " * (81 - len(line)) + "\n"
    else:
        line = line[:80] + "\n"
    return line

def _split_before_terminal(records: List[str]) -> Tuple[List[str], List[str]]:
    """Return (prefix, terminal_block) so we can insert ligand before END/ENDMDL/MASTER."""
    terminal = {"END", "ENDMDL", "MASTER"}
    i = len(records)
    # find beginning of trailing terminal block
    while i > 0:
        tok = records[i-1].strip().split()[:1]
        if tok and tok[0] in terminal:
            i -= 1
        else:
            break
    return records[:i], records[i:]

def _last_serial(lines: List[str]) -> int:
    mx = 0
    for ln in lines:
        if ln.startswith(("ATOM  ", "HETATM")):
            try:
                mx = max(mx, int(ln[6:11]))
            except Exception:
                pass
    return mx

def _infer_element_from_name(name: str) -> str:
    s = "".join(ch for ch in name.strip() if ch.isalpha()).upper()
    if len(s) >= 2 and s[:2] in _TWO_LETTER:
        return s[:2]
    return s[:1] or "C"

def _format_atom_name(name: str, elem: str) -> str:
    """
    PDB rule of thumb:
    - 1-letter element → right-justify 4-char atom name (element letter ends up at col 14).
    - 2-letter element → left-justify.
    """
    e = (elem or "").strip().upper()
    n = (name or e).strip()
    if len(e) == 1:
        return f"{n:>4}"[:4]
    else:
        return f"{n:<4}"[:4]

def _format_hetatm(
    serial: int, elem: str, atom_name: str,
    resname: str, chain: str, resseq: int,
    x: float, y: float, z: float,
    occ: float = 1.00, bfac: float = 0.00,
    alt: str = " ", icode: str = " "
) -> str:
    """
    Strict HETATM line (80 columns):
    1-6  HETATM
    7-11 serial
    13-16 atom name (formatted)
    17   altLoc
    18-20 resName (right-justified)
    22   chainID
    23-26 resSeq
    27   iCode
    31-38 x, 39-46 y, 47-54 z
    55-60 occupancy, 61-66 tempFactor
    77-78 element (right-justified)
    """
    e = (elem or "").strip().upper()
    if not e:
        e = _infer_element_from_name(atom_name)
    aname = _format_atom_name(atom_name, e)
    res = f"{(resname or 'LIG').upper():>3}"[:3]
    ch = (chain or " ")[:1]
    line = (
        f"HETATM{serial:5d} {aname}{alt[:1] if alt else ' '}"
        f"{res} {ch}{int(resseq):4d}{icode[:1] if icode else ' '}"
        f"   {x:8.3f}{y:8.3f}{z:8.3f}"
        f"{occ:6.2f}{bfac:6.2f}"
        f"{' ':10}{e:>2}{' ':2}"
    )
    return _ensure_nl80(line)

# ---------------- Robust SDF poses loader ----------------

def _load_poses_robust(sdf_path: str) -> Iterable[Chem.Mol]:
    """
    Yields RDKit Mol per pose from a GNINA multi-pose SDF.
    - Read with sanitize=False
    - Try sanitize; if it fails, rebuild bonds from coordinates.
    """
    from rdkit.Chem.rdDetermineBonds import DetermineConnectivity, DetermineBondOrders
    fsup = Chem.ForwardSDMolSupplier(sdf_path, removeHs=False, sanitize=False)
    for m in fsup:
        if m is None:
            continue
        m.UpdatePropertyCache(strict=False)
        try:
            Chem.SanitizeMol(m)
            yield m
            continue
        except Exception:
            pass
        # Rebuild from coords
        try:
            DetermineConnectivity(m)
            total_charge = sum(a.GetFormalCharge() for a in m.GetAtoms())
            DetermineBondOrders(
                m,
                charge=total_charge,
                allowChargedFragments=True,
                embedChiral=True,
                useAtomMap=False,
            )
            Chem.SanitizeMol(m)
            yield m
        except Exception:
            # Still yield unsanitized with coords; downstream just needs positions/element
            yield m

# ---------------- Complex writer (per pose) ----------------

def write_complexes_from_gnina_sdf(
    receptor_pdb: str,
    poses_sdf: str,
    out_prefix: str,
    ligand_resname: str = "LIG",
    chain_id: str = "A",
    resseq: int = 1,
    write_conect: bool = True,
    unique_atom_names: bool = True,
) -> List[str]:
    """
    Create one complex PDB per pose: receptor (as-is) + ligand HETATM (+ optional CONECT).
    Returns list of written file paths.
    """
    # read receptor text and split before terminal records
    with open(receptor_pdb, "r") as fh:
        rec_all = fh.readlines()
    rec_prefix, rec_terminal = _split_before_terminal(rec_all)
    base_serial = _last_serial(rec_prefix)

    os.makedirs(os.path.dirname(out_prefix) or ".", exist_ok=True)

    written: List[str] = []
    pose_idx = 0

    for mol in _load_poses_robust(poses_sdf):
        if mol.GetNumAtoms() == 0:
            continue
        pose_idx += 1

        conf = mol.GetConformer()
        serial = base_serial
        het_lines: List[str] = []
        idx2serial: Dict[int, int] = {}

        # Unique atom names per residue if requested
        elem_counts = defaultdict(int)
        def unique_name(sym: str) -> str:
            s = (sym or "X").upper()
            elem_counts[s] += 1
            return f"{s}{elem_counts[s]}"[:4]

        for a in mol.GetAtoms():
            i = a.GetIdx()
            p = conf.GetAtomPosition(i)
            elem = a.GetSymbol() or "C"
            name = unique_name(elem) if unique_atom_names else (elem or "X")
            serial += 1
            het_lines.append(
                _format_hetatm(
                    serial=serial, elem=elem, atom_name=name,
                    resname=ligand_resname, chain=chain_id, resseq=int(resseq),
                    x=p.x, y=p.y, z=p.z, occ=1.00, bfac=0.00
                )
            )
            idx2serial[i] = serial

        conect_lines: List[str] = []
        if write_conect:
            # Build neighbors and chunk in groups of 4 (PDB limit per line)
            neigh = defaultdict(list)
            for b in mol.GetBonds():
                i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
                si, sj = idx2serial.get(i), idx2serial.get(j)
                if si is None or sj is None:
                    continue
                neigh[si].append(sj)
                neigh[sj].append(si)
            for s, nbrs in neigh.items():
                for k in range(0, len(nbrs), 4):
                    chunk = nbrs[k:k+4]
                    line = "CONECT" + f"{s:5d}" + "".join(f"{n:5d}" for n in chunk)
                    conect_lines.append(_ensure_nl80(line))

        out_path = f"{out_prefix}_pose{pose_idx:03d}.pdb"
        with open(out_path, "w") as fh:
            fh.writelines(rec_prefix + het_lines + conect_lines + (rec_terminal if rec_terminal else ["END\n"]))
        written.append(out_path)

    return written

# -------------- Example direct run (edit paths) --------------
if __name__ == "__main__":
    receptor_pdb = "/content/rec.pdb"     # protein-only ATOM entries
    poses_sdf = "/content/Gnina_output/docked_ligand.sdf"       # GNINA multi-pose SDF
    out_prefix = "/content/Gnina_output/complex"            # writes complex_pose001.pdb, ...

    files = write_complexes_from_gnina_sdf(
        receptor_pdb, poses_sdf, out_prefix,
        ligand_resname="JQ1", chain_id="A", resseq=1,
        write_conect=True, unique_atom_names=True
    )
    print("[OK] wrote", len(files), "files")
    for p in files:
        print(" -", p)

[OK] wrote 9 files
 - /content/Gnina_output/complex_pose001.pdb
 - /content/Gnina_output/complex_pose002.pdb
 - /content/Gnina_output/complex_pose003.pdb
 - /content/Gnina_output/complex_pose004.pdb
 - /content/Gnina_output/complex_pose005.pdb
 - /content/Gnina_output/complex_pose006.pdb
 - /content/Gnina_output/complex_pose007.pdb
 - /content/Gnina_output/complex_pose008.pdb
 - /content/Gnina_output/complex_pose009.pdb


[11:55:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:55:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:55:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:55:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:55:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:55:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:55:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:55:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[11:55:42] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [ ]:
# Download the output folder
!zip -r output_folder.zip '/content/Gnina_output'

from google.colab import files
files.download('output_folder.zip')

updating: content/Gnina_output/ (stored 0%)
updating: content/Gnina_output/complex_pose009.pdb (deflated 75%)
updating: content/Gnina_output/complex_pose004.pdb (deflated 75%)
updating: content/Gnina_output/docked_ligand.sdf (deflated 83%)
updating: content/Gnina_output/complex_pose003.pdb (deflated 75%)
updating: content/Gnina_output/docking.log (deflated 55%)
updating: content/Gnina_output/complex_pose006.pdb (deflated 75%)
updating: content/Gnina_output/complex_pose001.pdb (deflated 75%)
updating: content/Gnina_output/complex_pose002.pdb (deflated 75%)
updating: content/Gnina_output/complex_pose007.pdb (deflated 75%)
updating: content/Gnina_output/complex_pose005.pdb (deflated 75%)
updating: content/Gnina_output/complex_pose008.pdb (deflated 75%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>